# Sharp-wave ripples and hippocampal replay in DANDI:000044

This notebook demonstrates two linked phenomena in rat dorsal CA1, end to end, from
streamed NWB files on the DANDI Archive:

1. **Sharp-wave ripples (SWRs)**: brief 140-200 Hz oscillations in the CA1 pyramidal
   layer, riding on a slow sharp wave generated by the CA3 input to stratum radiatum,
   during which the local population fires several times above its baseline rate.
2. **Replay**: within those ripples, place cells fire in sequences that recapitulate
   the trajectories the animal ran earlier, compressed roughly tenfold in time.

## Dataset

[DANDI:000044](https://dandiarchive.org/dandiset/000044), Grosmark & Buzsáki (2016),
*Diversity in neural firing dynamics supports both rigid and learned hippocampal
sequences* (Science 351:1440). Eight bilateral silicon-probe recordings (128 sites,
1250 Hz LFP plus spike-sorted units) from four Long-Evans rats. Every session has the
same three-part structure:

| epoch | content |
|---|---|
| PRE  | ~3 h rest/sleep in the familiar home cage |
| MAZE | ~35 min running for water reward on a **novel** maze in a **novel** room |
| POST | ~2-4 h rest/sleep back in the home cage |

That design gives the replay analysis a negative control that costs nothing: PRE sleep
was recorded before the animal had ever entered the novel maze, so the maze's place-field
template cannot be replayed there. Whatever significance rate the test returns in PRE is
the rate it returns on data that cannot contain the effect.

## Approach

Streaming access with `remfile` + a local disk cache (the files are 5-9 GB each and
only a handful of LFP channels are ever needed), `pynapple` for time-series handling,
tuning curves and peri-event analysis, and a hand-rolled memoryless Bayesian decoder
for the replay step so that the shuffles can reuse exactly the same code path.

In [1]:
import os

import matplotlib

matplotlib.use("Agg")  # this notebook is run headless; figures go to disk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pynapple as nap
from scipy.stats import chi2_contingency, mannwhitneyu
from tqdm.auto import tqdm

import analysis_lib as A
import swr_lib as L

os.makedirs("figures", exist_ok=True)
SESSION = "Achilles_10252013"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 10})

## 1. Load the session and inspect every data stream

`swr_lib.load_core` opens the NWB file over HTTP and wraps the spike times, epochs,
scored brain states and linearised position in pynapple objects. Nothing but metadata
is transferred at this point; the LFP is pulled lazily, one channel at a time.

In [2]:
d = L.load_core(SESSION)
units, epochs, states, position = d["units"], d["epochs"], d["states"], d["position"]
h5 = d["h5"]

print("epochs")
for k, v in epochs.items():
    print(f"  {k:12s} {v.start[0]:9.1f} - {v.end[-1]:9.1f} s  ({v.tot_length()/60:6.1f} min)")
print("brain states")
for k, v in states.items():
    print(f"  {k:8s} n={len(v):3d}  total {v.tot_length()/60:7.1f} min")
ct = units.get_info("cell_type")
print("units:", {c: int((ct == c).sum()) for c in np.unique(ct)},
      "| locations:", {c: int((units.get_info("location") == c).sum())
                       for c in np.unique(units.get_info("location"))})
print("LFP:", L.lfp_dataset(h5).shape, "at", L.LFP_FS, "Hz")
print(f"linearised position: {np.isfinite(position.d).sum()} valid of {len(position)} samples "
      f"(the archived linearisation only covers time on the track)")

epochs
  PREEpoch           0.0 -   18079.5 s  ( 301.3 min)
  MazeEpoch      18079.5 -   20147.0 s  (  34.5 min)
  POSTEpoch      20147.0 -   34861.1 s  ( 245.2 min)
brain states
  Awake    n= 62  total   279.1 min
  Non-REM  n= 60  total   264.8 min
  REM      n= 22  total    34.5 min
units: {'excitatory': 120, 'inhibitory': 17} | locations: {'lCA1': 65, 'rCA1': 72}
LFP: (43576379, 128) at 1250.0 Hz
linearised position: 10336 valid of 80762 samples (the archived linearisation only covers time on the track)


A first look at the whole session: population firing rate, a spike raster, and the
hypnogram. The three epochs are obvious in the firing rate, and non-REM dominates
both rest epochs.

In [3]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True,
                         gridspec_kw=dict(height_ratios=[1.1, 2, 1]))
allspk = nap.Ts(np.sort(np.concatenate([units[i].t for i in units.index])))
rate = allspk.count(10.0) / 10.0
axes[0].plot(rate.t / 60, rate.d, lw=0.6, color="k")
axes[0].set_ylabel("population rate\n(spikes/s)")
axes[0].set_title(f"{SESSION}: session overview (DANDI:000044, rat dorsal CA1)")
for name, col in [("PREEpoch", "#8ecae6"), ("MazeEpoch", "#ffb703"), ("POSTEpoch", "#90be6d")]:
    e = epochs[name]
    axes[0].axvspan(e.start[0] / 60, e.end[-1] / 60, color=col, alpha=0.3, zorder=0)
    axes[0].text((e.start[0] + e.end[-1]) / 120, axes[0].get_ylim()[1] * 0.92,
                 name.replace("Epoch", ""), ha="center", fontsize=10)
order = np.argsort(units.get_info("shank_id").values)
for row, ui in enumerate(np.array(units.index)[order]):
    t = units[ui].t
    t = t[:: max(1, len(t) // 3000)]
    axes[1].plot(t / 60, np.full_like(t, row), "|", ms=1.2,
                 color="tab:red" if ct[ui] == "inhibitory" else "k", alpha=0.5)
axes[1].set_ylabel("unit (sorted by shank)")
for i, (lab, col) in enumerate([("Awake", "#457b9d"), ("Non-REM", "#e63946"), ("REM", "#2a9d8f")]):
    for s, e in zip(states[lab].start, states[lab].end):
        axes[2].axvspan(s / 60, e / 60, ymin=i / 3, ymax=(i + 1) / 3, color=col, lw=0)
axes[2].set_yticks([1 / 6, 0.5, 5 / 6])
axes[2].set_yticklabels(["Awake", "Non-REM", "REM"])
axes[2].set_xlabel("time (min)")
axes[2].set_ylim(0, 1)
fig.tight_layout()
fig.savefig("figures/01_session_overview.png")
plt.close(fig)

The behaviour: the animal shuttles back and forth along the 1.6 m track, about 130
traversals at up to 1.2 m/s. The archived linearisation is defined only while the
animal is on the track proper, which is exactly the data the place-field step needs.

In [4]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(position.t, position.d, lw=0.7, color="k")
axes[0].set_ylabel("linearised position (m)")
axes[0].set_title(f"{SESSION}: running on the novel 1.6 m linear maze")
pos, vel, support = A.behaviour(position, 1.0 / 39.06263603480421)
axes[1].plot(pos.t, np.abs(vel.d), lw=0.6, color="tab:blue")
axes[1].set_ylabel("speed (m/s)")
axes[1].set_xlabel("time (s)")
fig.tight_layout()
fig.savefig("figures/01_maze_position.png")
plt.close(fig)

## 2. Choose a ripple channel

Ripples are largest in the CA1 pyramidal cell layer and fall off with distance from it.
Picking the channel with the most raw 140-230 Hz power would just pick the noisiest
site, so each channel is scored by the height of its ripple peak above a baseline
interpolated across the band (from 100-120 Hz and 280-320 Hz) in non-REM sleep.

In [5]:
rip_ch, ch_score = A.select_ripple_channel(h5, states["Non-REM"])
etab = d["nwbfile"].electrodes.to_dataframe()
shank = L.decode(etab["group_name"].values)
print(f"selected channel {rip_ch} on {shank[rip_ch]}")

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(ch_score, "o-", ms=3, color="tab:blue")
ax.axvline(rip_ch, color="r", ls="--", label=f"selected ch {rip_ch} ({shank[rip_ch]})")
ax.set_xlabel("LFP channel")
ax.set_ylabel("ripple bump\n(log$_{10}$ power above baseline)")
ax.set_title(f"{SESSION}: ripple-band spectral bump across the 128 recording sites (non-REM)")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig("figures/01_channel_survey.png")
plt.close(fig)

selected channel 10 on shank2


## 3. Detect sharp-wave ripples

The selected channel is band-pass filtered at 130-250 Hz, Hilbert-enveloped, smoothed
with an 8 ms Gaussian and z-scored **against non-REM sleep**, so that running theta
and its harmonics cannot drag the threshold around. An event needs a peak of 5 SD,
is delimited at 2 SD, must last 20-200 ms, and is rejected if it rides on a broadband
deflection larger than 12 SD (chewing and movement artefacts).

In [6]:
raw = L.lfp_dataset(h5)[:, rip_ch].astype(np.float32) * L.lfp_conversion(h5)
nrem_mask = A.nrem_mask_from(states, raw.size)
rip, rip_filt, rip_z = A.detect_ripples(raw, nrem_mask)
rip = A.tag_events(rip, states, epochs)
rip.to_csv("ripples.csv", index=False)
print(f"{len(rip)} ripples detected over {raw.size/L.LFP_FS/3600:.2f} h")
print(rip.groupby(["epoch", "state"]).size())

rate_rows = []
for lab in ["Non-REM", "REM", "Awake"]:
    for ep in ["PRE", "Maze", "POST"]:
        dur = states[lab].intersect(epochs[ep + "Epoch"]).tot_length()
        if dur < 60:
            continue
        cnt = int(((rip["state"] == lab) & (rip["epoch"] == ep)).sum())
        rate_rows.append(dict(epoch=ep, state=lab, n=cnt, minutes=dur / 60, rate_hz=cnt / dur))
rates = pd.DataFrame(rate_rows)
rates.to_csv("ripple_rates.csv", index=False)
print(rates.to_string(index=False))

8348 ripples detected over 9.68 h
epoch  state  
Maze   Awake       156
POST   Awake      1965
       Non-REM    2183
       other        18
PRE    Awake       892
       Non-REM    3125
       REM           1
       other         8
dtype: int64
epoch   state    n    minutes  rate_hz
  PRE Non-REM 3125 176.150000 0.295676
 POST Non-REM 2183  88.683333 0.410261
  PRE     REM    1  22.216667 0.000750
 POST     REM    0  12.300000 0.000000
  PRE   Awake  892 101.591667 0.146337
 Maze   Awake  156  34.458333 0.075453
 POST   Awake 1965 143.066667 0.228914


The rates are the first sanity check and they behave exactly as the literature says
they should: a few tenths of a hertz in non-REM sleep, essentially zero in REM (where
theta replaces the sharp-wave state), an intermediate rate during quiet waking, and
a higher non-REM rate in POST than in PRE.

### 3a. Ripple-triggered averages

Averaging the raw LFP on the ripple peak recovers the sharp wave; averaging a Morlet
wavelet transform (pynapple's `compute_wavelet_transform`) recovers a power increase
confined to 140-200 Hz and to about ±30 ms.

In [7]:
half = int(0.25 * L.LFP_FS)
sel = rip[(rip["epoch"] == "POST") & (rip["state"] == "Non-REM")]
peaks = (sel["peak_t"].values * L.LFP_FS).astype(int)
peaks = peaks[(peaks > half) & (peaks < raw.size - half)]
snips = np.stack([raw[p - half : p + half] for p in peaks])
snips_f = np.stack([rip_filt[p - half : p + half] for p in peaks])
tt = np.arange(-half, half) / L.LFP_FS * 1000

freqs = np.logspace(np.log10(20), np.log10(300), 60)
sub = snips[:: max(1, len(snips) // 500)]
wt = nap.compute_wavelet_transform(
    nap.TsdFrame(t=np.arange(2 * half) / L.LFP_FS, d=sub.T), freqs, fs=L.LFP_FS)
power = np.abs(np.asarray(wt)) ** 2  # (time, event, frequency)
spec = power.mean(axis=1).T
spec_z = spec / spec[:, : int(0.05 * L.LFP_FS)].mean(1, keepdims=True)

fig, axes = plt.subplots(3, 1, figsize=(7.5, 9), sharex=True,
                         gridspec_kw=dict(height_ratios=[1, 1, 1.4]))
m = snips.mean(0) * 1e3
se = snips.std(0) * 1e3 / np.sqrt(len(snips))
axes[0].plot(tt, m, color="k")
axes[0].fill_between(tt, m - se, m + se, color="k", alpha=0.3)
axes[0].set_ylabel("broadband LFP (mV)")
axes[0].set_title(f"Ripple-triggered average, POST non-REM (n={len(snips)}, ch {rip_ch})")
axes[1].plot(tt, snips_f.mean(0) * 1e3, color="tab:purple")
axes[1].set_ylabel("130-250 Hz (mV)")
im = axes[2].pcolormesh(tt, freqs, spec_z, shading="auto", cmap="magma")
axes[2].set_yscale("log")
axes[2].set_yticks([20, 50, 100, 200, 300])
axes[2].set_yticklabels([20, 50, 100, 200, 300])
axes[2].set_ylabel("frequency (Hz)")
axes[2].set_xlabel("time from ripple peak (ms)")
fig.colorbar(im, ax=axes[2], label="power / baseline", pad=0.01)
for a in axes:
    a.axvline(0, color="0.6", lw=0.7, ls="--", zorder=0)
fig.tight_layout()
fig.savefig("figures/02_ripple_triggered_average.png")
plt.close(fig)

### 3b. Event properties

In [8]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
nremsel = rip[rip["state"] == "Non-REM"]
axes[0, 0].hist(nremsel["duration"] * 1000, bins=40, color="tab:blue")
axes[0, 0].set_xlabel("duration (ms)")
axes[0, 0].set_ylabel("# ripples")
axes[0, 0].set_title(f"median {nremsel['duration'].median()*1000:.0f} ms")
axes[0, 1].hist(nremsel["peak_freq"], bins=40, color="tab:purple")
axes[0, 1].set_xlabel("intra-ripple frequency (Hz)")
axes[0, 1].set_title(f"median {nremsel['peak_freq'].median():.0f} Hz")
axes[0, 2].hist(nremsel["peak_z"], bins=np.arange(5, 30, 0.5), color="tab:green")
axes[0, 2].set_xlabel("peak envelope (z)")
axes[0, 2].set_title("event amplitude")
iri = np.diff(np.sort(nremsel["peak_t"].values))
axes[1, 0].hist(iri[iri < 5], bins=60, color="tab:orange")
axes[1, 0].set_xlabel("inter-ripple interval (s)")
axes[1, 0].set_ylabel("# intervals")
piv = rates.pivot_table(index="state", columns="epoch", values="rate_hz")
piv = piv.reindex(index=["Non-REM", "REM", "Awake"], columns=["PRE", "Maze", "POST"])
piv.plot.bar(ax=axes[1, 1], rot=0)
axes[1, 1].set_ylabel("ripple rate (Hz)")
axes[1, 1].set_title("rate by state and epoch")
axes[1, 1].legend(fontsize=8)
axes[1, 2].scatter(nremsel["duration"] * 1000, nremsel["peak_freq"], s=3, alpha=0.25)
axes[1, 2].set_xlabel("duration (ms)")
axes[1, 2].set_ylabel("intra-ripple frequency (Hz)")
fig.suptitle(f"{SESSION}: sharp-wave ripple properties ({len(nremsel)} non-REM events)")
fig.tight_layout()
fig.savefig("figures/02_ripple_properties.png")
plt.close(fig)

### 3c. The laminar signature

The strongest evidence that these are sharp-wave **ripples** rather than filter
artefacts is their depth profile. Averaging the ripple-triggered LFP across the ten
sites of one shank shows the ripple envelope peaking at the top of the shank and a
slow deflection that reverses polarity a few hundred micrometres deeper: positive in
the cell layer and stratum oriens, negative in stratum radiatum where the CA3 Schaffer
collateral input generates the sharp-wave current sink.

In [9]:
# use the shank that carries the detection channel, ordered superficial to deep
SHANK_CH = np.flatnonzero(shank == shank[rip_ch])
halfl = int(0.15 * L.LFP_FS)
i0 = int(epochs["POSTEpoch"].start[0] * L.LFP_FS)
i1 = int(epochs["POSTEpoch"].end[-1] * L.LFP_FS)
pk = (sel["peak_t"].values * L.LFP_FS).astype(int)
pk = pk[(pk - halfl > i0) & (pk + halfl < i1)]

if os.path.exists("laminar_profile.npy"):
    lam, lam_env = np.load("laminar_profile.npy")
else:
    lam = np.zeros((len(SHANK_CH), 2 * halfl))
    lam_env = np.zeros_like(lam)
    conv = L.lfp_conversion(h5)
    for k, ch in enumerate(tqdm(SHANK_CH, desc="laminar profile")):
        x = L.lfp_dataset(h5)[i0:i1, ch].astype(np.float32) * conv
        _, env = L.band_envelope(x, *A.DET_BAND)
        idx = pk - i0
        lam[k] = np.stack([x[p - halfl : p + halfl] for p in idx]).mean(0)
        # the envelope, not the filtered trace: ripple phase is not locked across
        # events, so averaging the oscillation itself would cancel
        lam_env[k] = np.stack([env[p - halfl : p + halfl] for p in idx]).mean(0)
        del x, env
    np.save("laminar_profile.npy", np.stack([lam, lam_env]))
ttl = np.arange(-halfl, halfl) / L.LFP_FS * 1000

fig, axes = plt.subplots(1, 3, figsize=(14, 6), gridspec_kw=dict(width_ratios=[1, 1, 1.15]))
step, step_e = np.abs(lam).max() * 1.1, lam_env.max() * 1.2
for k in range(len(SHANK_CH)):
    axes[0].plot(ttl, lam[k] * 1e3 - k * step * 1e3, color="k", lw=1)
    axes[0].text(-148, -k * step * 1e3 + step * 300, f"ch{SHANK_CH[k]}", va="bottom", fontsize=8)
    axes[1].plot(ttl, lam_env[k] * 1e3 - k * step_e * 1e3, color="tab:purple", lw=1)
    axes[1].text(-148, -k * step_e * 1e3 + step_e * 300, f"ch{SHANK_CH[k]}", va="bottom", fontsize=8)
axes[0].set_title(f"ripple-triggered average LFP\n({shank[rip_ch]}, superficial $\\rightarrow$ deep)")
axes[1].set_title("mean 130-250 Hz envelope\n(same events and depths)")
for a in axes[:2]:
    a.axvline(0, color="0.6", ls="--", lw=0.7)
    a.set_xlabel("time from ripple peak (ms)")
    a.set_yticks([])
axes[0].set_ylabel("depth along shank")
im = axes[2].imshow(lam * 1e3, aspect="auto", cmap="RdBu_r",
                    extent=[ttl[0], ttl[-1], len(SHANK_CH) - 0.5, -0.5],
                    vmin=-np.abs(lam).max() * 1e3, vmax=np.abs(lam).max() * 1e3)
axes[2].set_yticks(range(len(SHANK_CH)))
axes[2].set_yticklabels([f"ch{c}" for c in SHANK_CH], fontsize=8)
axes[2].axvline(0, color="k", ls="--", lw=0.7)
axes[2].set_xlabel("time from ripple peak (ms)")
axes[2].set_title("depth $\\times$ time (mV)")
fig.colorbar(im, ax=axes[2], label="mV", pad=0.02)
fig.suptitle(f"{SESSION}: laminar signature of the sharp wave-ripple (n={len(pk)} events)")
fig.tight_layout()
fig.savefig("figures/03_laminar_profile.png")
plt.close(fig)

### 3d. Single events and the population burst

In [10]:
pyr_ids = [i for i in units.index if ct[i] == "excitatory"]
int_ids = [i for i in units.index if ct[i] == "inhibitory"]
raster_order = list(pyr_ids) + list(int_ids)
examples = sel.sort_values("peak_z", ascending=False).head(200).sample(3, random_state=0)
examples = examples.sort_values("peak_t")

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey="row")
for col, r in enumerate(examples.itertuples()):
    t0, t1 = r.peak_t - 0.25, r.peak_t + 0.25
    a0, a1 = int(t0 * L.LFP_FS), int(t1 * L.LFP_FS)
    tt2 = (np.arange(a0, a1) / L.LFP_FS - r.peak_t) * 1000
    axes[0, col].plot(tt2, raw[a0:a1] * 1e3, color="k", lw=0.8, label="raw LFP")
    axes[0, col].plot(tt2, rip_filt[a0:a1] * 1e3 - 0.9, color="tab:purple", lw=0.8,
                      label="130-250 Hz")
    for ax in axes[:, col]:
        ax.axvspan((r.start - r.peak_t) * 1000, (r.stop - r.peak_t) * 1000,
                   color="tab:orange", alpha=0.18, zorder=0)
    axes[0, col].set_title(f"t = {r.peak_t:.1f} s, {r.peak_z:.1f} SD, {r.duration*1000:.0f} ms")
    epi = nap.IntervalSet(start=t0, end=t1)
    for row, ui in enumerate(raster_order):
        st = units[ui].restrict(epi).t
        if len(st):
            axes[1, col].plot((st - r.peak_t) * 1000, np.full(len(st), row), "|", ms=3,
                              color="tab:red" if ct[ui] == "inhibitory" else "k")
    axes[1, col].set_xlabel("time from ripple peak (ms)")
axes[0, 0].set_ylabel("LFP (mV)")
axes[0, 0].legend(fontsize=8, loc="upper left")
axes[1, 0].set_ylabel("unit (pyramidal, then interneuron)")
fig.suptitle(f"{SESSION}: example sharp-wave ripples with concurrent CA1 spiking")
fig.tight_layout()
fig.savefig("figures/03_example_ripples.png")
plt.close(fig)

In [11]:
ev_ts = nap.Ts(sel["peak_t"].values)
WIN, BIN = 0.5, 0.005


def psth(group):
    pe = nap.compute_perievent(group, ev_ts, (-WIN, WIN))
    window = nap.IntervalSet(-WIN, WIN)
    out = []
    for ui in pe.keys():
        c = np.asarray(pe[ui].count(BIN, window))  # (n_bins, n_events)
        out.append(c.sum(axis=1) / (BIN * c.shape[1]))
    return np.array(out), pe[list(pe.keys())[0]].count(BIN, window).t


pyr_psth, t_psth = psth(units[pyr_ids])
int_psth, _ = psth(units[int_ids])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for arr, lab, col in [(pyr_psth, "pyramidal", "k"), (int_psth, "interneuron", "tab:red")]:
    m = arr.mean(0)
    se = arr.std(0) / np.sqrt(len(arr))
    axes[0].plot(t_psth * 1000, m, color=col, label=f"{lab} (n={len(arr)})")
    axes[0].fill_between(t_psth * 1000, m - se, m + se, color=col, alpha=0.3)
axes[0].axvline(0, color="0.75", ls="--", lw=0.8, zorder=0)
axes[0].set_xlabel("time from ripple peak (ms)")
axes[0].set_ylabel("firing rate (spikes/s)")
axes[0].set_title("ripple-triggered firing rate")
axes[0].legend(fontsize=9)

base = pyr_psth[:, np.abs(t_psth) > 0.3].mean(1, keepdims=True) + 1e-9
norm = pyr_psth / base
srt = np.argsort(norm[:, np.abs(t_psth) < 0.05].mean(1))
im = axes[1].imshow(norm[srt], aspect="auto", cmap="magma", vmin=0, vmax=6,
                    extent=[t_psth[0] * 1000, t_psth[-1] * 1000, len(norm), 0])
axes[1].set_xlabel("time from ripple peak (ms)")
axes[1].set_ylabel("pyramidal cell (sorted by modulation)")
axes[1].set_title("per-cell rate / baseline")
fig.colorbar(im, ax=axes[1], label="rate / baseline", pad=0.02)

mod_pyr = norm[:, np.abs(t_psth) < 0.05].mean(1)
mod_int = (int_psth / (int_psth[:, np.abs(t_psth) > 0.3].mean(1, keepdims=True) + 1e-9))[
    :, np.abs(t_psth) < 0.05].mean(1)
axes[2].hist([mod_pyr, mod_int], bins=np.arange(0, 8, 0.4), stacked=True,
             color=["k", "tab:red"], label=["pyramidal", "interneuron"])
axes[2].axvline(1, color="0.5", ls="--")
axes[2].set_xlabel("in-ripple rate / baseline rate")
axes[2].set_ylabel("# cells")
axes[2].set_title(f"median gain: pyr {np.median(mod_pyr):.1f}x, int {np.median(mod_int):.1f}x")
axes[2].legend(fontsize=9)
fig.suptitle(f"{SESSION}: CA1 population activity during sharp-wave ripples (n={len(ev_ts)})")
fig.tight_layout()
fig.savefig("figures/03_ripple_spiking.png")
plt.close(fig)
print(f"pyramidal gain median {np.median(mod_pyr):.2f}x, "
      f"{100*(mod_pyr>2).mean():.0f}% of pyramidal cells more than double their rate")

pyramidal gain median 3.44x, 95% of pyramidal cells more than double their rate


## 4. Place fields on the novel maze

Tuning curves come from `pynapple.compute_tuning_curves` over 4 cm bins, restricted to
periods when the animal was tracked on the track and moving faster than 5 cm/s, and
computed separately for rightward and leftward runs because CA1 fields on a linear
track are strongly direction-selective.

Two cell selections are used, and it is worth keeping them apart. The **decoding
template** takes every pyramidal cell with a peak tuning-curve rate above 1 Hz and a
mean maze rate above 0.05 Hz: the decoder weights each cell by its own field, so it
does not need cells to be certified place cells, and a spatial-information cut tuned
on one session discards most of the population in the lower-yield ones. Skaggs spatial
information is reported separately as a description of how sharply tuned the population
is, using the conventional 0.5 bits/spike line.

In [12]:
POS_DT = 1.0 / 39.06263603480421
TRACK_LEN, N_BINS, SPEED_TH = 1.6, 40, 0.05

pyr = units[pyr_ids]
speed = nap.Tsd(t=pos.t, d=np.abs(vel.d), time_support=support)
run_ep = speed.threshold(SPEED_TH).time_support
right_ep = vel.threshold(SPEED_TH).time_support
left_ep = nap.Tsd(t=pos.t, d=-vel.d, time_support=support).threshold(SPEED_TH).time_support
print(f"running {run_ep.tot_length():.0f} s of {support.tot_length():.0f} s tracked "
      f"({right_ep.tot_length():.0f} s rightward, {left_ep.tot_length():.0f} s leftward)")

tc_all = A.place_fields(pyr, pos, run_ep, N_BINS, TRACK_LEN, POS_DT)
tc_right = A.place_fields(pyr, pos, right_ep, N_BINS, TRACK_LEN, POS_DT)
tc_left = A.place_fields(pyr, pos, left_ep, N_BINS, TRACK_LEN, POS_DT)
xbins = tc_all.index.values
occ_s = np.histogram(pos.restrict(run_ep).d, bins=N_BINS, range=(0, TRACK_LEN))[0] * POS_DT
si, mean_rate = A.skaggs_information(tc_all, occ_s)
peak_rate = tc_all.values.max(0)
is_place = (si > 0.5) & (peak_rate > 1.0)  # descriptive: classic place-cell criterion
keep_cell = A.select_template_cells(tc_all, mean_rate)  # what goes into the decoder
place_ids = np.array(tc_all.columns)[keep_cell]
si_ids = np.array(tc_all.columns)[is_place]
print(f"decoding template: {keep_cell.sum()} of {len(keep_cell)} pyramidal cells")
print(f"classic place-cell cut (SI > 0.5 bits/spike and peak > 1 Hz): {is_place.sum()} cells, "
      f"median SI {np.median(si):.2f} bits/spike")

running 244 s of 264 s tracked (131 s rightward, 112 s leftward)


decoding template: 88 of 120 pyramidal cells
classic place-cell cut (SI > 0.5 bits/spike and peak > 1 Hz): 49 cells, median SI 0.59 bits/spike


In [13]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8), gridspec_kw=dict(height_ratios=[1, 1.4]))
axes[0, 0].bar(xbins, occ_s, width=xbins[1] - xbins[0], color="0.5")
axes[0, 0].set_xlabel("position (m)")
axes[0, 0].set_ylabel("occupancy (s)")
axes[0, 0].set_title(f"running occupancy ({run_ep.tot_length():.0f} s total)")
axes[0, 1].hist(si, bins=30, color="tab:blue")
axes[0, 1].axvline(0.5, color="r", ls="--")
axes[0, 1].set_xlabel("spatial information (bits/spike)")
axes[0, 1].set_ylabel("# pyramidal cells")
axes[0, 1].set_title(f"{is_place.sum()} place cells of {len(si)}")
axes[0, 2].scatter(mean_rate, si, s=12, c=np.where(is_place, "tab:red", "0.6"))
axes[0, 2].set_xscale("log")
axes[0, 2].set_xlabel("mean firing rate on maze (Hz)")
axes[0, 2].set_ylabel("spatial information (bits/spike)")
pf = tc_all.values[:, is_place]
normf = pf / pf.max(0, keepdims=True)
srt = np.argsort(np.argmax(normf, axis=0))
im = axes[1, 0].imshow(normf[:, srt].T, aspect="auto", cmap="viridis",
                       extent=[0, TRACK_LEN, is_place.sum(), 0])
axes[1, 0].set_xlabel("position (m)")
axes[1, 0].set_ylabel("place cell (sorted by peak)")
axes[1, 0].set_title("normalised place fields (both directions)")
fig.colorbar(im, ax=axes[1, 0], pad=0.02)
for ax, tc, lab in [(axes[1, 1], tc_right, "rightward runs"), (axes[1, 2], tc_left, "leftward runs")]:
    p = tc.values[:, is_place]
    ax.imshow((p / np.maximum(p.max(0, keepdims=True), 1e-9))[:, srt].T, aspect="auto",
              cmap="viridis", extent=[0, TRACK_LEN, is_place.sum(), 0])
    ax.set_xlabel("position (m)")
    ax.set_title(lab + "\n(same cell order)")
fig.suptitle(f"{SESSION}: CA1 place fields on the novel 1.6 m linear maze")
fig.tight_layout()
fig.savefig("figures/04_place_fields.png")
plt.close(fig)

In [14]:
peak_pos = xbins[np.argmax(tc_all.values[:, is_place], axis=0)]
show = []
for target in [0.15, 0.4, 0.65, 0.9, 1.15, 1.45]:
    for c in np.argsort(np.abs(peak_pos - target)):
        if si_ids[c] not in show:
            show.append(si_ids[c])
            break
fig, axes = plt.subplots(2, 6, figsize=(16, 6), sharex=True,
                         gridspec_kw=dict(height_ratios=[1.4, 1]))
for j, ui in enumerate(show):
    sp = pyr[ui].restrict(run_ep)
    sp_pos = np.interp(sp.t, pos.t, pos.d)
    sp_dir = np.interp(sp.t, vel.t, vel.d) > 0
    axes[0, j].plot(sp_pos[sp_dir], sp.t[sp_dir] / 60, ".", ms=1.5, color="tab:blue", label="R")
    axes[0, j].plot(sp_pos[~sp_dir], sp.t[~sp_dir] / 60, ".", ms=1.5, color="tab:orange", label="L")
    axes[0, j].set_title(f"unit {ui}\nSI={si[list(tc_all.columns).index(ui)]:.2f} bits/spk", fontsize=9)
    axes[1, j].plot(xbins, tc_right[ui], color="tab:blue")
    axes[1, j].plot(xbins, tc_left[ui], color="tab:orange")
    axes[1, j].set_xlabel("position (m)")
axes[0, 0].set_ylabel("time (min)")
axes[0, 0].legend(fontsize=7, markerscale=5)
axes[1, 0].set_ylabel("rate (Hz)")
fig.suptitle(f"{SESSION}: example place cells (blue = rightward runs, orange = leftward)")
fig.tight_layout()
fig.savefig("figures/04_example_place_cells.png")
plt.close(fig)

## 5. Replay: Bayesian decoding inside ripples

Every candidate ripple is cut into 20 ms bins (with 25 ms of padding on each side),
kept only if it yields at least 5 bins and at least 5 active place cells, and decoded
with the memoryless Bayesian estimator

$$P(x \mid n) \;\propto\; P(x)\ \prod_i f_i(x)^{n_i}\ e^{-\tau \sum_i f_i(x)}$$

against both the rightward and the leftward template. Each event is scored by the
posterior-weighted correlation between decoded position and time, taking whichever
template gives the larger absolute value.

Significance requires beating **three** shuffles at p < 0.05, each of which destroys
something different while preserving the rest:

* **column cycle** — roll each time bin's posterior independently in space (keeps what
  each bin says about position, destroys the sequence);
* **place-field identity** — permute which cell owns which field and re-decode (keeps
  every field shape and every cell's in-event spike count, destroys the map);
* **time-bin order** — permute the bins within the event (keeps every bin's content,
  destroys their order).

Awake ripples on the maze only count if the animal was immobile, so that ongoing theta
sequences cannot masquerade as replay.

In [15]:
st_t, st_v = [], []
for b in A.tracking_blocks(position):
    st_t.append(position.t[b])
    st_v.append(np.abs(np.gradient(position.d[b], position.t[b])))
st_t, st_v = np.concatenate(st_t), np.concatenate(st_v)
j = np.clip(np.searchsorted(st_t, rip["peak_t"].values), 1, len(st_t) - 1)
moving = (np.abs(st_t[j] - rip["peak_t"].values) < 0.2) & (st_v[j] > SPEED_TH)

cand = rip[
    ((rip["epoch"] == "PRE") & (rip["state"] == "Non-REM"))
    | ((rip["epoch"] == "POST") & (rip["state"] == "Non-REM"))
    | ((rip["epoch"] == "Maze") & (rip["state"] == "Awake") & ~moving)
].reset_index(drop=True)
group = units[list(place_ids)]
counts, keep, nbins = A.build_event_counts(cand, group, place_ids)
print(f"{len(counts)} of {len(cand)} candidate ripples pass the spiking criteria")

dec = A.ReplayDecoder(counts, xbins,
                      {"right": tc_right[list(place_ids)].values.T,
                       "left": tc_left[list(place_ids)].values.T})
out = dec.run(n_shuffles=500, seed=1, progress=tqdm)

ev = cand.loc[keep].reset_index(drop=True)
for k in ["r", "slope", "direction", "p_column", "p_placefield", "p_timebin", "significant"]:
    ev[k] = out[k]
ev["n_bins"] = nbins
ev.to_csv("replay_events.csv", index=False)

for lab in ["PRE", "POST", "Maze"]:
    m = ev["epoch"] == lab
    s = int((m & ev["significant"]).sum())
    fwd = int((m & ev["significant"] & (ev["r"] > 0)).sum())
    print(f"  {lab:5s} {s:4d}/{int(m.sum()):5d} significant ({100*s/max(m.sum(),1):5.1f}%)  "
          f"forward {fwd}, reverse {s-fwd},  mean |r| {np.abs(ev.loc[m,'r']).mean():.3f}")

pre_m, post_m = ev["epoch"] == "PRE", ev["epoch"] == "POST"
tab = np.array([[int((ev["significant"] & pre_m).sum()), int((~ev["significant"] & pre_m).sum())],
                [int((ev["significant"] & post_m).sum()), int((~ev["significant"] & post_m).sum())]])
chi2, pval, _, _ = chi2_contingency(tab)
u, pu = mannwhitneyu(np.abs(ev.loc[post_m, "r"]), np.abs(ev.loc[pre_m, "r"]), alternative="greater")
print(f"\nPRE vs POST significant fraction: chi2 = {chi2:.1f}, p = {pval:.2e}")
print(f"POST |r| > PRE |r| (Mann-Whitney): U = {u:.0f}, p = {pu:.2e}")

2758 of 5464 candidate ripples pass the spiking criteria


shuffles:   0%|          | 0/500 [00:00<?, ?it/s]

  PRE     35/ 1829 significant (  1.9%)  forward 21, reverse 14,  mean |r| 0.366
  POST    52/  833 significant (  6.2%)  forward 31, reverse 21,  mean |r| 0.411
  Maze    33/   96 significant ( 34.4%)  forward 13, reverse 20,  mean |r| 0.598

PRE vs POST significant fraction: chi2 = 32.6, p = 1.15e-08
POST |r| > PRE |r| (Mann-Whitney): U = 853309, p = 3.22e-07


### 5a. Example replay events

The posterior sweeps linearly across the track within 100-180 ms, and the place-cell
raster underneath (cells ordered by field position) shows the same diagonal.

In [16]:
field_peak = xbins[np.argmax(tc_all[list(place_ids)].values, axis=0)]
cell_order = np.argsort(field_peak)
DIR_IDX = {"right": 0, "left": 1}
post_arrays = np.stack([out["posteriors"]["right"], out["posteriors"]["left"]])

sig_ev = ev[ev["significant"]]
picks = []
for epoch, want_fwd in [("POST", True), ("POST", False), ("Maze", True), ("Maze", False),
                        ("POST", True), ("Maze", False)]:
    sub = sig_ev[(sig_ev["epoch"] == epoch) & ((sig_ev["r"] > 0) == want_fwd)]
    sub = sub[~sub.index.isin(picks)]
    if len(sub):
        picks.append(sub["r"].abs().idxmax() if len(picks) % 2 == 0
                     else sub["r"].abs().sort_values().index[-2])
picks = list(dict.fromkeys(picks))[:6]

fig, axes = plt.subplots(2, len(picks), figsize=(3.1 * len(picks), 6.6), sharey="row")
for col, i in enumerate(picks):
    r = ev.loc[i]
    p = post_arrays[DIR_IDX[r["direction"]], i, : int(r["n_bins"])]
    nb = int(r["n_bins"])
    axes[0, col].pcolormesh(np.arange(nb + 1) * A.TAU * 1000,
                            np.r_[xbins - 0.02, xbins[-1] + 0.02], p.T,
                            cmap="magma", shading="auto")
    tc_ = (np.arange(nb) + 0.5) * A.TAU
    mt = (p * tc_[:, None]).sum() / p.sum()
    mx = (p * xbins[None, :]).sum() / p.sum()
    axes[0, col].plot(tc_ * 1000, mx + r["slope"] * (tc_ - mt), color="cyan", lw=1.5)
    axes[0, col].set_ylim(xbins[0] - 0.02, xbins[-1] + 0.02)
    axes[0, col].set_title(f"{r['epoch']} {'forward' if r['r']>0 else 'reverse'}\n"
                           f"r={r['r']:.2f}, {abs(r['slope']):.1f} m/s, {r['direction']} map",
                           fontsize=9)
    epi = nap.IntervalSet(start=r["start"] - A.PAD, end=r["start"] - A.PAD + nb * A.TAU)
    for row, ci in enumerate(cell_order):
        stt = units[place_ids[ci]].restrict(epi).t
        if len(stt):
            axes[1, col].plot((stt - (r["start"] - A.PAD)) * 1000, np.full(len(stt), row),
                              "|", ms=5, color="k")
    axes[1, col].set_xlim(0, nb * A.TAU * 1000)
    axes[1, col].set_xlabel("time in event (ms)")
axes[0, 0].set_ylabel("decoded position (m)")
axes[1, 0].set_ylabel("place cell\n(ordered by field position)")
fig.suptitle(f"{SESSION}: example significant replay events (posterior + place-cell raster)")
fig.tight_layout()
fig.savefig("figures/06_replay_examples.png")
plt.close(fig)

### 5b. Population summary

The bottom-middle panel is the honest part of the analysis: in PRE sleep, where no
replay of the novel maze can exist, the field-identity and time-bin p-values are close
to uniform, as a valid null should be, while the column-cycle shuffle on its own is
clearly liberal. That is why significance is required against all three.

In [17]:
epochs_lab = [("PRE", "PRE sleep\n(never saw maze)"), ("POST", "POST sleep"),
              ("Maze", "awake on maze\n(immobile)")]
cols = {"PRE": "#8ecae6", "POST": "#e63946", "Maze": "#ffb703"}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
ax = axes[0, 0]
fracs, cis, ns = [], [], []
for e, _ in epochs_lab:
    m = ev["epoch"] == e
    k, n = int(ev.loc[m, "significant"].sum()), int(m.sum())
    f = k / n
    fracs.append(f * 100)
    cis.append(1.96 * np.sqrt(f * (1 - f) / n) * 100)
    ns.append((k, n))
ax.bar(range(3), fracs, yerr=cis, color=[cols[e] for e, _ in epochs_lab], capsize=4)
ax.axhline(5, color="0.4", ls="--", lw=1)
ax.text(2.45, 5.6, "nominal 5%", fontsize=8, ha="right", color="0.3")
ax.set_xticks(range(3))
ax.set_xticklabels([l for _, l in epochs_lab], fontsize=9)
ax.set_ylabel("% of ripples with significant replay")
for i, (k, n) in enumerate(ns):
    ax.text(i, fracs[i] + cis[i] + 1.2, f"{k}/{n}", ha="center", fontsize=8)
ax.set_title("significant replay (3 shuffles, all p < 0.05)")

ax = axes[0, 1]
for e, lab in epochs_lab:
    m = ev["epoch"] == e
    ax.hist(np.abs(ev.loc[m, "r"]), bins=np.arange(0, 1.02, 0.05), density=True,
            histtype="step", lw=2, color=cols[e], label=lab.replace("\n", " "))
ax.set_xlabel("|weighted correlation|")
ax.set_ylabel("density")
ax.legend(fontsize=8)
ax.set_title("replay score of every candidate ripple")

ax = axes[0, 2]
w = 0.35
for j, (e, _) in enumerate(epochs_lab):
    m = (ev["epoch"] == e) & ev["significant"]
    ax.bar(j - w / 2, int((ev.loc[m, "r"] > 0).sum()), w, color=cols[e])
    ax.bar(j + w / 2, int((ev.loc[m, "r"] < 0).sum()), w, color=cols[e], alpha=0.5, hatch="//")
ax.set_xticks(range(3))
ax.set_xticklabels([l for _, l in epochs_lab], fontsize=9)
ax.set_ylabel("# significant events")
ax.set_title("forward (solid) vs reverse (hatched)")

ax = axes[1, 0]
sp = np.abs(ev.loc[ev["significant"], "slope"])
ax.hist(sp, bins=np.arange(0, 30, 1.5), color="0.4")
ax.axvline(np.median(sp), color="r", ls="--")
ax.set_xlabel("replay speed |slope| (m/s)")
ax.set_ylabel("# significant events")
ax.set_title(f"virtual speed, median {np.median(sp):.1f} m/s\n(real running peaks near 1 m/s)")

ax = axes[1, 1]
for name, key in [("column cycle", "p_column"), ("field identity", "p_placefield"),
                  ("time-bin order", "p_timebin")]:
    ax.hist(ev.loc[ev["epoch"] == "PRE", key], bins=np.arange(0, 1.01, 0.05),
            histtype="step", lw=1.8, density=True, label=f"{name} (PRE)")
ax.axhline(1, color="0.5", ls=":", lw=1)
ax.set_xlabel("shuffle p-value, PRE sleep")
ax.set_ylabel("density")
ax.legend(fontsize=8)
ax.set_title("null behaviour in the control epoch")

ax = axes[1, 2]
post_ev = ev[ev["epoch"] == "POST"].copy()
post_ev["dt_min"] = (post_ev["peak_t"] - epochs["MazeEpoch"].end[-1]) / 60
edges = np.arange(0, post_ev["dt_min"].max() + 30, 30)
mids, fr, err = [], [], []
for a, b in zip(edges[:-1], edges[1:]):
    m = (post_ev["dt_min"] >= a) & (post_ev["dt_min"] < b)
    if m.sum() < 20:
        continue
    k, n = int(post_ev.loc[m, "significant"].sum()), int(m.sum())
    mids.append((a + b) / 2)
    fr.append(100 * k / n)
    err.append(100 * 1.96 * np.sqrt((k / n) * (1 - k / n) / n))
ax.errorbar(mids, fr, yerr=err, marker="o", color="#e63946")
pre_rate = 100 * ev.loc[ev["epoch"] == "PRE", "significant"].mean()
ax.axhline(pre_rate, color="#8ecae6", lw=2, label=f"PRE baseline ({pre_rate:.1f}%)")
ax.set_xlabel("minutes after leaving the maze")
ax.set_ylabel("% significant replay")
ax.legend(fontsize=8)
ax.set_title("replay across POST sleep")

fig.suptitle(f"{SESSION}: replay of the novel linear maze during sharp-wave ripples")
fig.tight_layout()
fig.savefig("figures/06_replay_summary.png")
plt.close(fig)

## 6. All eight sessions

The single-session numbers are only worth as much as their reproducibility, so the
same pipeline is run over every session in the dandiset. Ripple statistics are
reported for all eight; replay is scored for the five linear-maze sessions only,
because on a circular maze the linearised coordinate wraps and a weighted correlation
against a straight line is not the right statistic.

`multi_session.py` writes its results to `multi_session/`; if they are already there
this cell just loads them (the sweep takes about half an hour, most of it transfer).

In [18]:
import multi_session as MS

if not os.path.exists("multi_session/session_summary.csv"):
    MS.sweep()
sess = pd.read_csv("multi_session/session_summary.csv")
rates_all = pd.read_csv("multi_session/ripple_rates_all_sessions.csv")
ev_all = pd.read_csv("multi_session/replay_events_all_sessions.csv")
print(sess[["session", "maze", "n_units", "n_ripples", "nrem_rate",
            "median_duration_ms", "median_freq_hz"]].to_string(index=False))

          session                             maze  n_units  n_ripples  nrem_rate  median_duration_ms  median_freq_hz
Achilles_10252013 1.6mLinearMazeLinearizedPosition      137       8348   0.334047                50.4      162.645811
Achilles_11012013   CircularMazeLinearizedPosition      104       7812   0.340562                51.2      164.871281
  Cicero_09012014 1.6mLinearMazeLinearizedPosition       73       6987   0.338433                49.6      160.436894
  Cicero_09102014   CircularMazeLinearizedPosition      105       7440   0.395871                48.0      163.710062
  Cicero_09172014   2mLinearMazeLinearizedPosition       72       6566   0.369715                47.2      163.666615
  Gatsby_08022013 1.6mLinearMazeLinearizedPosition       80       6752   0.331201                47.2      156.952263
  Gatsby_08282013   CircularMazeLinearizedPosition       51       7829   0.314575                53.6      161.097218
   Buddy_06272013 1.6mLinearMazeLinearizedPosition      

In [19]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
short = [s.split("_")[0][:3] + "\n" + s.split("_")[1] for s in sess["session"]]

ax = axes[0, 0]
ax.bar(range(len(sess)), sess["nrem_rate"], color=np.where(sess["circular"], "0.7", "tab:blue"))
ax.set_xticks(range(len(sess)))
ax.set_xticklabels(short, fontsize=7, rotation=45, ha="right")
ax.set_ylabel("non-REM ripple rate (Hz)")
ax.set_title("ripple rate per session\n(grey = circular maze)")

ax = axes[0, 1]
for col, (lab, c) in enumerate([("Non-REM", "#e63946"), ("REM", "#2a9d8f"), ("Awake", "#457b9d")]):
    v = rates_all[rates_all["state"] == lab].groupby("session")["n"].sum() / \
        rates_all[rates_all["state"] == lab].groupby("session")["seconds"].sum()
    ax.plot(np.full(len(v), col) + np.random.default_rng(0).normal(0, 0.05, len(v)), v,
            "o", color=c, ms=7)
ax.set_xticks(range(3))
ax.set_xticklabels(["non-REM", "REM", "awake"])
ax.set_ylabel("ripple rate (Hz)")
ax.set_title("rate by brain state, one point per session")

ax = axes[0, 2]
ax.scatter(sess["median_duration_ms"], sess["median_freq_hz"], s=60,
           c=np.where(sess["circular"], "0.7", "tab:blue"))
for i, s in enumerate(short):
    ax.annotate(s.replace("\n", " "), (sess["median_duration_ms"][i], sess["median_freq_hz"][i]),
                fontsize=6, xytext=(5, -8 if i % 2 else 5), textcoords="offset points")
ax.set_xlabel("median duration (ms)")
ax.set_ylabel("median intra-ripple frequency (Hz)")
ax.set_title("event properties per session")

lin = sess.dropna(subset=["frac_PRE"]).reset_index(drop=True)
ax = axes[1, 0]
for i, r in lin.iterrows():
    ax.plot([0, 1], [100 * r["frac_PRE"], 100 * r["frac_POST"]], "o-", color="0.4", ms=6)
ax.plot([0, 1], [100 * lin["sig_PRE"].sum() / lin["n_PRE"].sum(),
                 100 * lin["sig_POST"].sum() / lin["n_POST"].sum()],
        "o-", color="#e63946", lw=3, ms=10, label="pooled")
ax.set_xticks([0, 1])
ax.set_xticklabels(["PRE sleep", "POST sleep"])
ax.set_ylabel("% ripples with significant replay")
ax.axhline(5, color="0.6", ls="--", lw=1)
ax.legend(fontsize=8)
ax.set_title("PRE vs POST, five linear-maze sessions")

ax = axes[1, 1]
pooled = []
for lab, c in [("PRE", "#8ecae6"), ("POST", "#e63946"), ("Maze", "#ffb703")]:
    k = int(lin[f"sig_{lab}"].sum())
    n = int(lin[f"n_{lab}"].sum())
    pooled.append((lab, k, n))
    f = k / n
    ax.bar(len(pooled) - 1, 100 * f, yerr=100 * 1.96 * np.sqrt(f * (1 - f) / n), color=c, capsize=4)
    ax.text(len(pooled) - 1, 100 * f + 2, f"{k}/{n}", ha="center", fontsize=8)
ax.set_xticks(range(3))
ax.set_xticklabels(["PRE", "POST", "awake maze"])
ax.axhline(5, color="0.4", ls="--", lw=1)
ax.set_ylabel("% significant replay")
ax.set_title("pooled across sessions")

ax = axes[1, 2]
for lab, c in [("PRE", "#8ecae6"), ("POST", "#e63946"), ("Maze", "#ffb703")]:
    m = ev_all["epoch"] == lab
    ax.hist(np.abs(ev_all.loc[m, "r"]), bins=np.arange(0, 1.02, 0.05), density=True,
            histtype="step", lw=2, color=c, label=lab)
ax.set_xlabel("|weighted correlation|")
ax.set_ylabel("density")
ax.legend(fontsize=8)
ax.set_title("pooled replay scores")

fig.suptitle("DANDI:000044: sharp-wave ripples and replay across all eight sessions")
fig.tight_layout()
fig.savefig("figures/07_multi_session.png")
plt.close(fig)

tab_all = np.array([[int(lin["sig_PRE"].sum()), int(lin["n_PRE"].sum() - lin["sig_PRE"].sum())],
                    [int(lin["sig_POST"].sum()), int(lin["n_POST"].sum() - lin["sig_POST"].sum())]])
chi2a, pa, _, _ = chi2_contingency(tab_all)
print(f"pooled PRE vs POST: {tab_all[0,0]}/{tab_all[0].sum()} vs "
      f"{tab_all[1,0]}/{tab_all[1].sum()}, chi2 = {chi2a:.1f}, p = {pa:.2e}")
for lab, k, n in pooled:
    print(f"  {lab:5s} {k:4d}/{n:5d} = {100*k/n:.1f}%")

pooled PRE vs POST: 78/4734 vs 100/3730, chi2 = 10.3, p = 1.31e-03
  PRE     78/ 4734 = 1.6%
  POST   100/ 3730 = 2.7%
  Maze    97/  893 = 10.9%


## Conclusions

**Sharp-wave ripples.** Events detected on the best CA1 pyramidal-layer channel have
the full set of properties the phenomenon is defined by: a median duration near 50 ms
and an intra-ripple frequency near 160 Hz; a rate of a few tenths of a hertz in non-REM
sleep and effectively zero in REM; a ripple-band power increase confined to 140-200 Hz
and to ±30 ms around the peak; a laminar profile in which the oscillation is largest at
the cell layer while the accompanying slow sharp wave reverses polarity in stratum
radiatum; and a population burst in which pyramidal cells roughly triple their firing
rate and interneurons more than double theirs.

**Replay.** Decoding the place-cell population inside those ripples against the maze's
place-field template recovers sequential sweeps across the track at about ten times the
animal's running speed. The rate of significant events is highest during awake
immobility on the maze itself, intermediate in POST sleep, and lowest in PRE sleep,
which is the epoch that cannot contain replay of a maze the animal has not yet seen.
The awake events are biased towards reverse order and the sleep events are not, which
is the split reported in the literature for reward-site versus offline replay.

The single largest caveat is that the sleep effect is a difference between small
percentages: a few per cent of ripples in POST versus one to two per cent in PRE. The
effect survives pooling across five sessions, but any single event should be read as a
statistical claim about the population, not as a certainty about that event.